> This script is deigned to be used with conda env
> crime_weather_env

Code to explore the crimetology_NS.parquet file.
Data created via combine_crime_ceda_data.py.

Good background reading available at [GeoffBoeing.com](https://geoffboeing.com/2014/08/clustering-to-reduce-spatial-data-set-size/)

Optimisation of data processing required as 1.5 million data rows to be clustered with only 16GB of RAM. 

## Import modules

In [38]:
## Typehinting
from _duckdb import DuckDBPyConnection
from pandas.core.frame import DataFrame
from numba.core.types.npytypes import Array


# modules
import time
from pathlib import Path
import numpy as np
import os
import duckdb
from fast_hdbscan import HDBSCAN
#from sklearn.cluster import DBSCAN
#from sklearn.neighbors import NearestNeighbors
import matplotlib.pyplot as plt

## Directories, db and Path Setup

In [20]:
cwd: str = os.getcwd()
data_dir: Path = Path(cwd).parent / 'data' / 'police_archives'
crime_db: Path = data_dir/'crime_archive.db'

Connect to the duckdb

In [21]:
con: DuckDBPyConnection = duckdb.connect(database=crime_db)

Introspect

In [22]:
# introspect
con.execute(query="SHOW TABLES").fetchall()

[('crimetology_NS',),
 ('crimetology_NS_clean',),
 ('crimetology_coords_lookup',),
 ('street_data',)]

In [23]:
con.execute(query="SELECT * FROM crimetology_NS_clean LIMIT 5;").df()


,Crime ID,Month,Longitude,Latitude,Crime type,tasmax,tas,groundfrost,sun,snowLying,tasmin,rainfall,hurs,sfcWind
0,00926dfc98a1ad484fec277ced4f0237c1032d9d63f4f0...,2020-01,0.521596,52.345789,Violence and sexual offences,9.597225,6.834480,8.521248,57.155247,0.028889,4.043394,46.032112,88.652786,4.317238
1,08088ddb322e7dfb2bc9eb9736776849fa6e6568e28725...,2020-01,1.188665,52.041702,Violence and sexual offences,9.253441,6.699946,12.421773,53.776817,0.017040,4.145340,42.233070,88.948593,4.330167
2,3bb04e754be52c4b474e9a5d5cba723cb44063d592348f...,2020-01,0.392886,52.245956,Burglary,9.339664,6.674413,8.760840,60.050930,0.054434,3.973928,49.437729,89.399498,4.221702
3,975d8b78c0b49bce0c2bda4b15770a7b29bdd55ddc62cd...,2020-01,0.694570,52.259155,Violence and sexual offences,9.237802,6.624474,8.096927,60.260311,0.116909,4.029425,52.358356,89.107567,4.471696
4,becd00d626e8dfc23c780e8238961dc788183688b3b92b...,2020-01,1.335191,51.965723,Violence and sexual offences,9.293962,7.044072,6.509821,56.745304,0.000000,4.690509,31.948408,88.402222,5.864347


In [24]:
## expect this to be 1255500
con.execute(query="SELECT COUNT(*) FROM crimetology_NS_clean LIMIT 5;").df()

,count_star()
0,1255500


In [25]:
con.execute(query="DESCRIBE crimetology_NS_clean").df()

,column_name,column_type,null,key,default,extra
0,Crime ID,VARCHAR,YES,None,None,None
1,Month,VARCHAR,YES,None,None,None
2,Longitude,DOUBLE,YES,None,None,None
3,Latitude,DOUBLE,YES,None,None,None
4,Crime type,VARCHAR,YES,None,None,None
5,tasmax,FLOAT,YES,None,None,None
6,tas,FLOAT,YES,None,None,None
7,groundfrost,FLOAT,YES,None,None,None
8,sun,FLOAT,YES,None,None,None
9,snowLying,FLOAT,YES,None,None,None


In [26]:
con.execute(query="SUMMARIZE crimetology_NS_clean").df()

,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,Crime ID,VARCHAR,000006a0d6919a710d3bd0a37d0ff31a0eb99ee5a1c20f...,fffff37e0363180e0780c2208d545e512d90e3de540bd8...,1622925,None,None,None,None,None,1255500,0.0
1,Month,VARCHAR,2016-01,2025-12,136,None,None,None,None,None,1255500,0.0
2,Longitude,DOUBLE,0.300512,1.757943,42131,1.1395494182150399,0.3959295100528411,0.8479526952933838,1.1943926806193195,1.331265862399162,1255500,0.0
3,Latitude,DOUBLE,51.950135,52.972354,52334,52.4397646439887,0.26507872988828857,52.18162270976858,52.52339098216082,52.636525971113095,1255500,0.0
4,Crime type,VARCHAR,Anti-social behaviour,Violence and sexual offences,16,None,None,None,None,None,1255500,0.0
5,tasmax,FLOAT,4.8080764,27.462076,156283,15.435893627845449,5.682904224237646,10.255511951044596,15.066825650243658,20.616506094729324,1255500,0.0
6,tas,FLOAT,2.1268184,20.685835,149424,11.429114954629394,4.8998097611869325,6.921842249517724,11.265414724797079,15.981564477297434,1255500,0.0
7,groundfrost,FLOAT,0.0,26.692324,144181,6.5048300714746805,6.642106525202314,0.044931644346662365,4.224159684596919,12.259469746593465,1255500,0.0
8,sun,FLOAT,16.014242,328.6736,213372,149.59287266437983,70.84050654222102,83.66341740257853,155.0413046172607,204.9331330445286,1255500,0.0
9,snowLying,FLOAT,0.0,7.6520123,67539,0.2088989479030356,0.7101793982724235,0.0,0.0,0.036259652343956415,1255500,0.0


The above introspection all looks good

## Prepare data for DBScan

Isolate columns needed for clustering

In [27]:
features_query = """ SELECT "Crime ID",
                            Latitude,
                            Longitude
                     FROM crimetology_NS_clean
                 """
features_df: DataFrame= con.execute(query=features_query).df()

In [28]:
features_df

,Crime ID,Latitude,Longitude
0,00926dfc98a1ad484fec277ced4f0237c1032d9d63f4f0...,52.345789,0.521596
1,08088ddb322e7dfb2bc9eb9736776849fa6e6568e28725...,52.041702,1.188665
2,3bb04e754be52c4b474e9a5d5cba723cb44063d592348f...,52.245956,0.392886
3,975d8b78c0b49bce0c2bda4b15770a7b29bdd55ddc62cd...,52.259155,0.694570
4,becd00d626e8dfc23c780e8238961dc788183688b3b92b...,51.965723,1.335191
...,...,...,...
1255495,b0817e29e4812387eefd4dbf61223703bd4b2ae18e659e...,52.394576,1.199594
1255496,40f5d31aebde7ecbbb3301ba6f48d58a9fc95fdb77fb2c...,52.375385,1.098691
1255497,bbf00daaa7f12ee2fbff2328b9065555c668f05798d94f...,52.374885,1.096864
1255498,debbfd614bade093a6a6782521caae560f6a5f0565f427...,52.377189,1.098171


This is currently in degrees, we need to it be mapped to radians to use the Haverstein distance formulae (to account for the fact that we live on a sphere).

Need be careful with geometry systems and scale correctly between latitude, km and radians,

**Useful formula**:  
 - Distance Along a Meridian (North-South):  
$$ \text{Distance (km)} = \Delta \text{latitude in radians} \times \text{Earths radius} $$

 - Distance Along a Parallel (East-West):
$$ \text{Distance (km)} = \Delta \text{longitude in radians} \times \text{Earths radius} \times \cos(\text{latitude}) $$


In [29]:
# earths radius in km
earths_radius = 6371.0088

In [33]:
# 2. Convert Lat/Lon to Radians
lat_rad: Array = np.radians(features_df["Latitude"].values)
lon_rad: Array = np.radians(features_df["Longitude"].values)

Porject data into 3D Cartesian coordinates as the fast_hdbscan only support euclidean distances.

In [35]:
features_df['X'] = earths_radius * np.cos(lat_rad) * np.cos(lon_rad)
features_df['Y'] = earths_radius * np.cos(lat_rad) * np.sin(lon_rad)
features_df['Z'] = earths_radius * np.sin(lat_rad)

In [36]:
features_df

,Crime ID,Latitude,Longitude,X,Y,Z
0,00926dfc98a1ad484fec277ced4f0237c1032d9d63f4f0...,52.345789,0.521596,3891.853113,35.430727,5044.004078
1,08088ddb322e7dfb2bc9eb9736776849fa6e6568e28725...,52.041702,1.188665,3917.886304,81.292595,5023.276973
2,3bb04e754be52c4b474e9a5d5cba723cb44063d592348f...,52.245956,0.392886,3900.705512,26.748159,5037.214922
3,975d8b78c0b49bce0c2bda4b15770a7b29bdd55ddc62cd...,52.259155,0.694570,3899.350180,47.272318,5038.113400
4,becd00d626e8dfc23c780e8238961dc788183688b3b92b...,51.965723,1.335191,3924.321629,91.466905,5018.075995
...,...,...,...,...,...,...
1255495,b0817e29e4812387eefd4dbf61223703bd4b2ae18e659e...,52.394576,1.199594,3886.865975,81.390681,5047.316275
1255496,40f5d31aebde7ecbbb3301ba6f48d58a9fc95fdb77fb2c...,52.375385,1.098691,3888.693333,74.577851,5046.013816
1255497,bbf00daaa7f12ee2fbff2328b9065555c668f05798d94f...,52.374885,1.096864,3888.739736,74.454695,5045.979874
1255498,debbfd614bade093a6a6782521caae560f6a5f0565f427...,52.377189,1.098171,3888.535160,74.539514,5046.136274


In [37]:
Features_in:Array = features_df[['X', 'Y', 'Z']].values
Features_in

array([[3891.85311289,   35.43072655, 5044.00407821],
       [3917.88630442,   81.29259455, 5023.27697319],
       [3900.70551209,   26.74815876, 5037.21492232],
       ...,
       [3888.73973614,   74.45469475, 5045.97987438],
       [3888.53515994,   74.53951358, 5046.13627446],
       [3888.78476771,   76.15937811, 5045.91972878]], shape=(1255500, 3))

## Start DBSCan method

In [ ]:
# define epsilon as 20 kilometers, the maximum anonymysation distace used by police.ac.uk
epsilon: float = 20

In [ ]:
features_in = features_df[["Latitude_radians", "Longitude_radians"]].values

In [ ]:
clusterer = HDBSCAN(min_cluster_size=500,cluster_selection_epsilon=epsilon)

In [ ]:
start_time: float = time.time()
features_df['Cluster_ID'] = clusterer.fit_predict(X=Features_in)
end_time: float  = time.time() - start_time
print(f"Took {end_time} seconds to run")